# Dual MPC Two-Pass Comparison

This notebook compares `Round-1 Local MPC` against `Dual 2-Pass MPC`, shows stage progress during the dual rollout, reuses the `compare.ipynb` metric tables, and visualizes dual prices, power balance, battery/SoC, and net-load changes.

The evaluation window is resolved from the same reference compare bundle as `compare.ipynb`, so the test dates stay aligned automatically.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "configs").exists():
    repo_root = repo_root.parent
if not (repo_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import configs as configs_pkg
from configs import compose_experiment_config
from scripts.utils import grid_notebook_workflow as grid_nb
from scripts.utils import dual_distributed_notebook_helpers as dual_nb
from scripts.utils.experiment_notebook_utils import resolve_madrl_model_root
from scripts.utils.project_paths import project_root as resolve_project_root

configs_pkg = importlib.reload(configs_pkg)
grid_nb = importlib.reload(grid_nb)
dual_nb = importlib.reload(dual_nb)


In [ ]:
PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_ROOT = None

COMPARE_REFERENCE_PREDICTION_MODE = "normal"
COMPARE_REFERENCE_LABEL = "MADRL + No Safety"
NOTEBOOK_PREDICTION_MODE = "perfect"

round1_label = "Round-1 Local MPC"
dual_label = "Dual 2-Pass MPC"
max_bisect_iters = 6
show_progress = True
dual_plot_episode_idx = 0
dual_context_steps = 2

DRL_RUN_SPECS = {
    "MADRL + No Safety": {"algorithm": "MATD3", "experiment_name": "train_base", "model_root": None},
    "MADRL + Safety Penalty": {"algorithm": "MATD3", "experiment_name": "train_base_safe", "model_root": None},
    "MADRL + Safety Projection": {
        "algorithm": "MATD3_SAFE_POC",
        "experiment_name": "train_projection_safe",
        "model_root": None,
    },
}

collect_mpc_rollout = grid_nb.collect_mpc_rollout
compare_rollout_metrics = grid_nb.compare_rollout_metrics
build_compare_economic_table = grid_nb.build_compare_economic_table
build_compare_safety_table = grid_nb.build_compare_safety_table
build_compare_warning_banner = grid_nb.build_compare_warning_banner
plot_power_balance_comparison = grid_nb.plot_power_balance_comparison
plot_battery_power_and_soc_comparison = grid_nb.plot_battery_power_and_soc_comparison
plot_net_load_comparison = grid_nb.plot_net_load_comparison

collect_dual_two_pass_rollout = dual_nb.collect_dual_two_pass_rollout
build_dual_window_diagnostic_frame = dual_nb.build_dual_window_diagnostic_frame
build_dual_first_step_summary_frame = dual_nb.build_dual_first_step_summary_frame


In [ ]:
resolved_model_roots = {}
for label, spec in DRL_RUN_SPECS.items():
    resolved_model_roots[label] = resolve_madrl_model_root(
        algorithm=spec["algorithm"],
        prediction_mode=COMPARE_REFERENCE_PREDICTION_MODE,
        experiment_name=spec["experiment_name"],
        model_root=spec.get("model_root"),
        root=PROJECT_ROOT,
        checkpoint_root=CHECKPOINT_ROOT,
    )

bundles = grid_nb.validate_compare_model_bundles(resolved_model_roots)
reference_bundle = bundles[COMPARE_REFERENCE_LABEL]
reference_experiment = reference_bundle["experiment_controls"]
reference_data = reference_bundle["data_controls"]
reference_battery = reference_bundle["battery_controls"]
reference_train = reference_bundle["train_controls"]

cfg = compose_experiment_config(
    profile=reference_train.get("profile", "base"),
    algorithm="MATD3",
    model_family=reference_train.get("model_family", "mlp"),
    data_dir=DATA_DIR,
    device=reference_experiment.get("device_request"),
    runtime_mode=reference_experiment.get("runtime_mode", "performance"),
    seed=int(reference_experiment.get("seed", 0)),
    require_cuda=reference_experiment.get("require_cuda"),
)
grid_nb.apply_notebook_experiment_settings(
    cfg,
    prediction_mode=NOTEBOOK_PREDICTION_MODE,
    test_start_date=reference_data.get("test_start_date"),
    test_end_date=reference_data.get("test_end_date"),
    agent_profiles=reference_data["agent_profiles"],
    agent_bus_ids=reference_data.get("agent_bus_ids"),
    load_scale=reference_data.get("load_scale"),
    pv_scale=reference_data.get("pv_scale"),
    battery_controls=reference_battery,
    future_horizon=reference_data.get("future_horizon"),
    train_year=reference_data.get("train_year"),
    test_year=reference_data.get("test_year"),
)

display(
    pd.Series(
        {
            "compare_reference_label": COMPARE_REFERENCE_LABEL,
            "compare_reference_prediction_mode": COMPARE_REFERENCE_PREDICTION_MODE,
            "notebook_prediction_mode": NOTEBOOK_PREDICTION_MODE,
            "test_start_date": cfg.data.test_start_date,
            "test_end_date": cfg.data.test_end_date,
            "test_year": cfg.data.test_year,
            "future_horizon": cfg.env.future_horizon,
            "agent_profiles": cfg.data.agent_profiles,
        },
        name="dual_notebook_compare_aligned_window",
    )
)


In [ ]:
def _select_dual_heatmap_frame(window_df: pd.DataFrame, *, episode_idx: int = 0, context_steps: int = 2) -> pd.DataFrame:
    episode_df = window_df.loc[window_df["episode_idx"] == int(episode_idx)].copy()
    if episode_df.empty:
        return episode_df
    active_steps = sorted(episode_df.loc[episode_df["dual_active"], "step"].astype(int).unique().tolist())
    if not active_steps:
        return episode_df.sort_values(["step", "horizon_step"]).reset_index(drop=True)
    min_step = int(episode_df["step"].min())
    max_step = int(episode_df["step"].max())
    selected_steps: set[int] = set()
    for step in active_steps:
        selected_steps.update(range(max(min_step, step - int(context_steps)), min(max_step, step + int(context_steps)) + 1))
    return episode_df.loc[episode_df["step"].isin(sorted(selected_steps))].sort_values(["step", "horizon_step"]).reset_index(drop=True)


def plot_dual_lambda_diagnostics(summary_df: pd.DataFrame, *, episode_idx: int = 0, figsize: tuple[float, float] = (18.0, 10.0)):
    episode_df = summary_df.loc[summary_df["episode_idx"] == int(episode_idx)].copy()
    if episode_df.empty:
        raise ValueError(f"No dual summary rows found for episode_idx={episode_idx}.")
    episode_df = episode_df.sort_values("step").reset_index(drop=True)
    timestamps = pd.to_datetime(episode_df["timestamp"])

    figure, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)

    active_mask = episode_df["dual_active_first_step"].astype(bool) | episode_df["dual_first_step_infeasible_export"].astype(bool)
    active_df = episode_df.loc[active_mask].copy()
    if active_df.empty:
        axes[0].plot(timestamps, np.zeros(len(episode_df), dtype=np.float32), color="#cbd5e1", linewidth=1.2)
        axes[0].text(0.5, 0.5, "No active dual-allocation steps in this episode.", transform=axes[0].transAxes, ha="center", va="center")
    else:
        markerline, stemlines, baseline = axes[0].stem(
            pd.to_datetime(active_df["timestamp"]),
            active_df["dual_lambda_first_step"].astype(float),
            linefmt="#2563eb",
            markerfmt="o",
            basefmt=" ",
        )
        plt.setp(stemlines, linewidth=1.6, color="#2563eb")
        plt.setp(markerline, markersize=6, markerfacecolor="#2563eb", markeredgecolor="#2563eb")
        infeasible_df = episode_df.loc[episode_df["dual_first_step_infeasible_export"].astype(bool)].copy()
        if not infeasible_df.empty:
            axes[0].scatter(pd.to_datetime(infeasible_df["timestamp"]), np.zeros(len(infeasible_df)), color="#dc2626", marker="x", s=60, label="surrogate infeasible")
            axes[0].legend(loc="upper right")
    axes[0].set_ylabel("lambda")
    axes[0].set_title(f"Dual lambda diagnostics - episode {episode_idx}")
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(timestamps, episode_df["dual_export_deficit_first_step_kw"].astype(float), color="#dc2626", linewidth=1.5, label="Export deficit (first step)")
    axes[1].plot(timestamps, episode_df["dual_delta_first_step_total_kw"].astype(float), color="#2563eb", linewidth=1.5, linestyle="--", label="Allocated delta (first step)")
    axes[1].set_ylabel("kW")
    axes[1].set_title("First-step deficit vs allocated net-load delta")
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(loc="upper right")

    axes[2].plot(timestamps, episode_df["dual_surrogate_trafo_relief_first_step_kw"].astype(float), color="#16a34a", linewidth=1.5, label="Surrogate relief (first step)")
    axes[2].plot(timestamps, episode_df["dual_pf_trafo_relief_kw"].astype(float), color="#7c3aed", linewidth=1.5, linestyle="--", label="PF relief")
    axes[2].plot(timestamps, episode_df["dual_surrogate_pf_relief_error_kw"].astype(float), color="#f59e0b", linewidth=1.2, linestyle=":", label="Surrogate - PF error")
    axes[2].set_ylabel("kW")
    axes[2].set_title("Surrogate vs PF relief")
    axes[2].grid(True, alpha=0.25)
    axes[2].legend(loc="upper right")
    axes[2].set_xlabel("Timestamp")

    figure.tight_layout()
    return figure


def plot_dual_resource_diagnostics(summary_df: pd.DataFrame, *, episode_idx: int = 0, figsize: tuple[float, float] = (18.0, 12.0)):
    episode_df = summary_df.loc[summary_df["episode_idx"] == int(episode_idx)].copy()
    if episode_df.empty:
        raise ValueError(f"No dual summary rows found for episode_idx={episode_idx}.")
    episode_df = episode_df.sort_values("step").reset_index(drop=True)
    timestamps = pd.to_datetime(episode_df["timestamp"])

    figure, axes = plt.subplots(4, 1, figsize=figsize, sharex=True)

    axes[0].plot(timestamps, episode_df["dual_export_deficit_first_step_kw"].astype(float), color="#dc2626", linewidth=1.4, label="Export deficit")
    axes[0].plot(timestamps, episode_df["dual_delta_first_step_total_kw"].astype(float), color="#2563eb", linewidth=1.5, linestyle="--", label="Requested delta")
    axes[0].plot(timestamps, episode_df["dual_round2_achieved_netload_lift_first_step_total_kw"].astype(float), color="#16a34a", linewidth=1.5, label="Round-2 achieved lift")
    axes[0].set_ylabel("kW")
    axes[0].set_title("Requested net-load lift vs achieved lift")
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(loc="upper right")

    total_headroom = (
        episode_df["dual_delta_battery_headroom_first_step_total_kw"].astype(float)
        + episode_df["dual_delta_curtail_headroom_first_step_total_kw"].astype(float)
    )
    axes[1].plot(timestamps, episode_df["dual_delta_battery_headroom_first_step_total_kw"].astype(float), color="#7c3aed", linewidth=1.5, label="Battery headroom")
    axes[1].plot(timestamps, episode_df["dual_delta_curtail_headroom_first_step_total_kw"].astype(float), color="#ea580c", linewidth=1.5, label="Curtailment headroom")
    axes[1].plot(timestamps, total_headroom, color="#111827", linewidth=1.3, linestyle=":", label="Total headroom")
    axes[1].set_ylabel("kW")
    axes[1].set_title("First-step flexibility headroom split")
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(loc="upper right")

    axes[2].plot(timestamps, episode_df["dual_round1_pv_curtail_first_step_total_kw"].astype(float), color="#94a3b8", linewidth=1.5, label="Round-1 curtailment")
    axes[2].plot(timestamps, episode_df["dual_round2_pv_curtail_first_step_total_kw"].astype(float), color="#dc2626", linewidth=1.5, label="Round-2 curtailment")
    axes[2].set_ylabel("kW")
    axes[2].set_title("First-step curtailment before vs after dual coordination")
    axes[2].grid(True, alpha=0.25)
    axes[2].legend(loc="upper right")

    axes[3].plot(timestamps, episode_df["dual_round1_pv_utilization_first_step"].astype(float), color="#94a3b8", linewidth=1.5, label="Round-1 PV utilization")
    axes[3].plot(timestamps, episode_df["dual_round2_pv_utilization_first_step"].astype(float), color="#16a34a", linewidth=1.5, label="Round-2 PV utilization")
    axes[3].set_ylabel("p.u.")
    axes[3].set_ylim(0.0, 1.05)
    axes[3].set_title("First-step PV utilization before vs after dual coordination")
    axes[3].grid(True, alpha=0.25)
    axes[3].legend(loc="upper right")
    axes[3].set_xlabel("Timestamp")

    figure.tight_layout()
    return figure


def plot_dual_lambda_heatmap(window_df: pd.DataFrame, *, episode_idx: int = 0, context_steps: int = 2, figsize: tuple[float, float] = (16.0, 5.0)):
    selected = _select_dual_heatmap_frame(window_df, episode_idx=episode_idx, context_steps=context_steps)
    if selected.empty:
        raise ValueError(f"No dual window rows found for episode_idx={episode_idx}.")
    pivot = selected.pivot(index="horizon_step", columns="step", values="dual_lambda").sort_index(axis=0).sort_index(axis=1)
    matrix = np.ma.masked_invalid(pivot.to_numpy(dtype=float))
    figure, axis = plt.subplots(1, 1, figsize=figsize)
    image = axis.imshow(matrix, aspect="auto", origin="lower", cmap="viridis")
    axis.set_title(f"Dual lambda heatmap - episode {episode_idx}")
    axis.set_xlabel("Real step")
    axis.set_ylabel("Horizon step")
    axis.set_xticks(np.arange(pivot.shape[1]))
    axis.set_xticklabels([str(int(step)) for step in pivot.columns], rotation=45, ha="right")
    axis.set_yticks(np.arange(pivot.shape[0]))
    axis.set_yticklabels([str(int(step)) for step in pivot.index])
    colorbar = figure.colorbar(image, ax=axis)
    colorbar.set_label("lambda")
    figure.tight_layout()
    return figure



In [ ]:
print("Starting Round-1 Local MPC...")
round1_rollout = collect_mpc_rollout(
    cfg,
    prediction_mode=NOTEBOOK_PREDICTION_MODE,
    label=round1_label,
)
print("Done:", round1_rollout.meta.get("controller", round1_label))

print("Starting Dual 2-Pass MPC...")
dual_rollout = collect_dual_two_pass_rollout(
    cfg,
    prediction_mode=NOTEBOOK_PREDICTION_MODE,
    label=dual_label,
    max_bisect_iters=max_bisect_iters,
    show_progress=True,
)
print("Done:", dual_rollout.meta.get("controller", dual_label))

rollouts = [round1_rollout, dual_rollout]
metrics_df = compare_rollout_metrics(*rollouts)
economic_table = build_compare_economic_table(metrics_df)
safety_table = build_compare_safety_table(metrics_df)
dual_window_df = build_dual_window_diagnostic_frame(dual_rollout)
dual_first_step_df = build_dual_first_step_summary_frame(dual_rollout)


In [ ]:
display(build_compare_warning_banner(*rollouts))
display(metrics_df)
display(economic_table)
display(safety_table)

dual_runtime_summary = pd.Series(
    {
        "n_real_steps": int(len(dual_first_step_df)),
        "n_active_dual_steps": int(dual_first_step_df["dual_active_first_step"].astype(bool).sum()) if not dual_first_step_df.empty else 0,
        "n_first_step_infeasible_export": int(dual_first_step_df["dual_first_step_infeasible_export"].astype(bool).sum()) if not dual_first_step_df.empty else 0,
        "n_flexibility_insufficient": int(dual_first_step_df["flexibility_insufficient"].astype(bool).sum()) if not dual_first_step_df.empty else 0,
        "n_import_only_unsupported": int(dual_first_step_df["import_only_unsupported"].astype(bool).sum()) if not dual_first_step_df.empty else 0,
    },
    name="dual_runtime_summary",
)
display(dual_runtime_summary)


In [ ]:
display(plot_dual_lambda_diagnostics(dual_first_step_df, episode_idx=dual_plot_episode_idx))
display(plot_dual_resource_diagnostics(dual_first_step_df, episode_idx=dual_plot_episode_idx))
display(plot_dual_lambda_heatmap(dual_window_df, episode_idx=dual_plot_episode_idx, context_steps=dual_context_steps))



In [ ]:
display(plot_power_balance_comparison(*rollouts))
display(plot_battery_power_and_soc_comparison(*rollouts))


In [ ]:
display(plot_net_load_comparison(*rollouts))


In [ ]:
dual_summary_columns = [
    "timestamp",
    "dual_lambda_first_step",
    "dual_export_deficit_first_step_kw",
    "dual_delta_first_step_total_kw",
    "dual_round2_achieved_netload_lift_first_step_total_kw",
    "dual_delta_battery_headroom_first_step_total_kw",
    "dual_delta_curtail_headroom_first_step_total_kw",
    "dual_round1_pv_curtail_first_step_total_kw",
    "dual_round2_pv_curtail_first_step_total_kw",
    "dual_round1_pv_utilization_first_step",
    "dual_round2_pv_utilization_first_step",
    "dual_pf_trafo_relief_kw",
    "dual_surrogate_pf_relief_error_kw",
    "dual_first_step_infeasible_export",
    "flexibility_insufficient",
]
display(dual_first_step_df.loc[:, dual_summary_columns].head(20))

dual_window_columns = [
    "timestamp",
    "step",
    "horizon_step",
    "dual_lambda",
    "dual_export_deficit_kw",
    "dual_delta_total_kw",
    "dual_battery_headroom_total_kw",
    "dual_curtail_headroom_total_kw",
    "dual_round2_achieved_netload_lift_total_kw",
    "dual_surrogate_trafo_relief_kw",
    "dual_pf_trafo_relief_kw",
    "dual_surrogate_pf_relief_error_kw",
    "dual_active",
    "dual_infeasible_export",
]
display(
    _select_dual_heatmap_frame(dual_window_df, episode_idx=dual_plot_episode_idx, context_steps=dual_context_steps)
    .loc[:, dual_window_columns]
    .head(50)
)

